# 01 函数逼近问题概览

第二章讨论的是插值：给定节点值，构造函数严格通过每个节点。第三章讨论更宽的任务：用一个简单函数空间去近似复杂函数或离散数据。

本章的基本问题可以写成：给定一个有限维空间

$$
V_n=\operatorname{span}\{\phi_0,\phi_1,\ldots,\phi_n\},
$$

在其中寻找

$$
p_n(x)=\sum_{k=0}^n c_k\phi_k(x),
$$

使误差 $f-p_n$ 在某种意义下尽量小。

这和插值的区别在于：插值先指定节点条件，逼近先指定函数空间和误差范数。空间和范数选得不同，“最好”的近似函数也会不同。


## 三类问题的差别

**插值**适合查表值和精确采样。它的典型条件是

$$
p(x_i)=y_i.
$$

如果数据无噪声，这个条件很自然；如果数据含噪声，强制穿过所有点可能会把噪声也当成真实信号。

**函数逼近**面对的是连续函数。例如给定

$$
f(x)=e^x\cos 3x,
$$

希望在

$$
V_n=\operatorname{span}\{1,x,x^2,\dots,x^n\}
$$

或 Chebyshev、Legendre 等正交多项式空间中找到 $p_n$，使 $f-p_n$ 在指定范数下尽量小。

**曲线拟合**面对的是离散观测数据。常见模型是

$$
p_n(x)=c_0+c_1x+\cdots+c_nx^n,
$$

并最小化残差平方和

$$
\sum_i (y_i-p_n(x_i))^2.
$$

因此，插值强调节点条件，函数逼近强调函数空间，曲线拟合强调数据残差。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import polynomial_eval, polynomial_least_squares

rng = np.random.default_rng(2026)
x_data = np.linspace(-1.0, 1.0, 18)
y_true = np.sin(2 * np.pi * x_data)
y_noisy = y_true + 0.12 * rng.normal(size=x_data.size)

x_eval = np.linspace(-1.0, 1.0, 400)
coefficients = polynomial_least_squares(x_data, y_noisy, degree=5)
y_fit = polynomial_eval(coefficients, x_eval)

plt.figure(figsize=(8, 4.5))
plt.scatter(x_data, y_noisy, color="black", label="含噪声数据")
plt.plot(x_eval, np.sin(2 * np.pi * x_eval), color="gray", label="真实函数")
plt.plot(x_eval, y_fit, label="五次最小二乘拟合")
plt.title("拟合允许残差，而不是强制通过每个点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.legend()
plt.tight_layout()
plt.show()


## 误差范数

逼近问题必须先说明“接近”的意义。常见选择有三种。

一致范数：

$$
\|f-g\|_\infty=\max_{x\in[a,b]} |f(x)-g(x)|.
$$

它关注最坏点误差，适合需要整个区间误差都受控的问题。

平方范数：

$$
\|f-g\|_2=\left(\int_a^b |f(x)-g(x)|^2 w(x)\,dx\right)^{1/2}.
$$

它来自内积

$$
\langle f,g\rangle_w=\int_a^b f(x)g(x)w(x)\,dx,
$$

强调平均意义下的误差。

离散二范数：

$$
\|r\|_2=\left(\sum_{i=1}^{m}r_i^2\right)^{1/2}.
$$

它对应最小二乘拟合。这里 $r_i=y_i-p(x_i)$ 是第 $i$ 个数据点上的残差。

平方范数下的最佳逼近具有清楚的几何意义：误差必须与逼近空间正交。这一投影观点会贯穿 Legendre 逼近和最小二乘拟合。


In [ ]:
def infinity_norm_error(f, g, x_grid):
    return np.max(np.abs(f(x_grid) - g(x_grid)))

# 教学式离散二范数：先构造残差，再取平方和开方。
residual = y_noisy - polynomial_eval(coefficients, x_data)
discrete_l2 = np.sqrt(np.sum(residual**2))

print(f"离散残差二范数: {discrete_l2:.3e}")
print(f"拟合多项式系数（升幂）: {coefficients}")


## 本章阅读顺序

本章后续内容按以下顺序展开：

1. 正交多项式与函数逼近：Chebyshev 和 Legendre；
2. 最小二乘曲线拟合：离散数据、Vandermonde 矩阵、过拟合；
3. Padé 有理逼近：从 Taylor 系数构造有理函数；
4. Fourier/FFT、自适应逼近和最佳一致逼近作为后续扩展框架。

这一章和第二章的关系是：Chebyshev 节点仍然出现，但重点不再是“插值公式”，而是“函数逼近工具”。

阅读时建议始终问三个问题：

* 当前使用的逼近空间是什么？
* 当前最小化的是哪一种误差？
* 当前算法是否数值稳定？


## 小结

* 插值要求精确通过节点；逼近要求在函数空间中整体接近；拟合允许离散残差。
* 误差范数决定“最好”的含义。
* 平方范数下的最佳逼近可以看作正交投影。
* 正交多项式把投影系数计算简化为逐项内积。
* 最小二乘拟合是离散数据上的投影问题。
* Padé 逼近用有理函数改善 Taylor 多项式的局限。
